In [0]:
%pip install rapidfuzz 'mlxtend<0.24' fastapi uvicorn nest_asyncio streamlit 'scipy<1.14' 'torch>=2.3.0'
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import re
import os
import warnings
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rapidfuzz import fuzz, process


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('future.no_silent_downcasting', True)

plt.style.use('seaborn-v0_8-muted')
sns.set_theme(style='whitegrid')

print('✅ Imports e configurações carregados com sucesso.')

✅ Imports e configurações carregados com sucesso.


In [0]:


def carregar_e_auditar_dados():
    tabelas_alvo = ['TAB_VENDA_ITEM', 'TAB_VENDA', 'TAB_PRODUTO']
    datasets_finais = {}
    
    # 📌 Configuração do Catálogo onde os dados foram salvos
    catalog_name = "workspace"
    schema_name = "bertoferramentas"

    print('=' * 80)
    print(f"📋 ESCOPO DA ANÁLISE: {', '.join(tabelas_alvo)}")
    print('=' * 80)

    for tabela in tabelas_alvo:
        print(f'\n▶️  PROCESSANDO: {tabela}')
        tabela_virtual = f"{catalog_name}.{schema_name}.{tabela.lower()}"
        
        try:
            # Consome a tabela virtualizada e converte para Pandas
            df = spark.table(tabela_virtual).toPandas()
            # Converte nomes de colunas para minúsculo
            df.columns = [col.lower() for col in df.columns]
            
            colunas_originais = list(df.columns)
            print(f'   📊 Campos identificados: {len(colunas_originais)}')

            # Limpa nulos textuais
            df = df.replace(r'^\s*$', np.nan, regex=True).replace('None', np.nan).replace('nan', np.nan)
            total_rows = len(df)

            campos_integros = []
            campos_para_remover = []

            for col in colunas_originais:
                nulos = df[col].isnull().sum()
                distintos = df[col].nunique()

                if nulos == 0:
                    campos_integros.append(col)

                if nulos == total_rows:
                    campos_para_remover.append((col, '100% Nulo'))
                elif distintos <= 1:
                    val = df[col].dropna().unique() if distintos == 1 else 'Vazio'
                    campos_para_remover.append((col, f"Constante ('{val}') — sem variância"))

            print(f'   ✅ Campos 100% preenchidos: {len(campos_integros)}')

            if campos_para_remover:
                print('   ❌ Campos removidos:')
                for col_rem, motivo in campos_para_remover:
                    print(f'      - {col_rem}: {motivo}')
            else:
                print('   ✨ Nenhum campo removido.')

            # Remove colunas desnecessárias
            cols_remover = [c for c, _ in campos_para_remover]
            df_final = df.drop(columns=cols_remover)
            datasets_finais[tabela] = df_final

            print(f'   📄 Resultado: {len(df_final.columns)} colunas × {total_rows} linhas.')

        except Exception as e:
            print(f'   💥 Erro ao carregar ou processar a tabela {tabela_virtual}: {e}')

    return datasets_finais

# Executa a auditoria apenas nos dados já virtualizados
datasets = carregar_e_auditar_dados()

📋 ESCOPO DA ANÁLISE: TAB_VENDA_ITEM, TAB_VENDA, TAB_PRODUTO

▶️  PROCESSANDO: TAB_VENDA_ITEM
   📊 Campos identificados: 18
   ✅ Campos 100% preenchidos: 15
   ✨ Nenhum campo removido.
   📄 Resultado: 18 colunas × 65490 linhas.

▶️  PROCESSANDO: TAB_VENDA
   📊 Campos identificados: 30
   ✅ Campos 100% preenchidos: 28
   ✨ Nenhum campo removido.
   📄 Resultado: 30 colunas × 31543 linhas.

▶️  PROCESSANDO: TAB_PRODUTO
   📊 Campos identificados: 29
   ✅ Campos 100% preenchidos: 1
   ❌ Campos removidos:
      - calcula_estoque: Constante ('['1']') — sem variância
      - sincronizado: Constante ('['0']') — sem variância
      - validade_ndias: Constante ('['1']') — sem variância
   📄 Resultado: 26 colunas × 18936 linhas.


In [0]:
# ============================================================
# CÉLULA 6 — SANEAMENTO: REMOÇÃO DE VENDAS ZERADAS
# ============================================================

def filtrar_vendas_invalidas(datasets):
    if 'TAB_VENDA' not in datasets:
        print('❌ TAB_VENDA não encontrada.')
        return datasets

    df_vendas = datasets['TAB_VENDA'].copy()
    col_valor  = 'ven_cur_subtotal'
    df_vendas[col_valor] = pd.to_numeric(df_vendas[col_valor], errors='coerce')

    total_original = len(df_vendas)
    df_limpo = df_vendas[
        (df_vendas[col_valor] > 0) & df_vendas[col_valor].notna()
    ]
    total_atual  = len(df_limpo)
    qtd_excluida = total_original - total_atual
    pct          = (qtd_excluida / total_original) * 100

    datasets['TAB_VENDA'] = df_limpo

    print('=' * 60)
    print('🛡️  SANEAMENTO: VENDAS ZERADAS')
    print('=' * 60)
    print(f'   Registros originais  : {total_original}')
    print(f'   Excluídos (valor≤0)  : {qtd_excluida} ({pct:.2f}%)')
    print(f'   Dataset válido       : {total_atual} registros')

    return datasets


datasets = filtrar_vendas_invalidas(datasets)

🛡️  SANEAMENTO: VENDAS ZERADAS
   Registros originais  : 31543
   Excluídos (valor≤0)  : 294 (0.93%)
   Dataset válido       : 31249 registros


In [0]:
# ============================================================
# CÉLULA 10 — CRIAÇÃO DO PRODUTO MESTRE (REGEX + FUZZY)
# ============================================================
import re
import pandas as pd
from rapidfuzz import process, fuzz

def criar_produto_mestre(datasets, threshold=82):
    """
    threshold : similaridade mínima (0-100) para dois mestres serem fundidos.
                82 é um bom ponto de partida para dados de varejo.
                Calibre usando a célula de diagnóstico anterior.
    """
    if 'TAB_PRODUTO' not in datasets:
        print('❌ TAB_PRODUTO não encontrada.')
        return datasets

    df_prod = datasets['TAB_PRODUTO'].copy()

    # ── ETAPA 1: Regex ──────────────────────────────────────
    def extrair_mestre_regex(txt):
        txt = str(txt).upper()
        
        # 1. Remove frações de polegada (ex: 1/2, 3/4, 1.1/2, 2 1/4)
        txt = re.sub(r'\d+[\.,]?\d*\s*/\s*\d+', ' ', txt)
        
        # 2. Remove dimensões com X (ex: 50X60, 4 X 2, 5.0x45, 12X12)
        txt = re.sub(r'\d+[\.,]?\d*\s*X\s*\d+[\.,]?\d*', ' ', txt)
        
        # 3. Remove palavras que contêm qualquer número grudado (ex: 25MM, 110V, P120, C64)
        txt = re.sub(r'\b[A-Z]*\d+[A-Z]*\b', ' ', txt)
        
        # 4. Remove unidades órfãs (todas as medidas, potências e embalagens)
        # Se algum "MM" ou "VOLT" ficar solto após tirar o número, ele morre aqui.
        unidades = (
            r'\b(MM|MILIMETRO|MILIMETROS|CM|CENTIMETRO|CENTIMETROS|M|METRO|METROS|'
            r'G|GR|GRAMA|GRAMAS|KG|KILO|KILOS|QUILO|QUILOS|QUILOGRAMA|QUILOGRAMAS|'
            r'L|LT|LITRO|LITROS|ML|MILILITRO|MILILITROS|POL|POLEGADA|POLEGADAS|'
            r'V|VOLT|VOLTS|W|WATT|WATTS|A|AMPERE|AMPERES|UN|UNID|UNIDADE|UNIDADES|'
            r'PC|PÇ|PCA|PÇA|PECA|PECAS|CX|CAIXA|CAIXAS|PCT|PACOTE|PACOTES|'
            r'JG|JOGO|PAR|PARES|KIT|FR|FRASCO|ROLO|ROLOS|BD|BALDE)\b'
        )
        txt = re.sub(unidades, ' ', txt)
        
        # 5. Remove pontuações e traços (ex: -, ., ())
        txt = re.sub(r'[^\w\s]', ' ', txt)
        
        # Divide as palavras e limpa espaços vazios
        palavras = txt.split()
        
        # Retorna as 3 primeiras palavras 
        return ' '.join(palavras[:3]) if len(palavras) >= 3 else ' '.join(palavras)

    df_prod['pro_txt_mestre'] = df_prod['pro_txt_descricao'].apply(extrair_mestre_regex)
    mestres_pos_regex = df_prod['pro_txt_mestre'].unique().tolist()

    print('=' * 70)
    print(f'🏗️  PRODUTO MESTRE — REGEX + FUZZY (threshold={threshold})')
    print('=' * 70)
    print(f'   SKUs originais          : {len(df_prod)}')
    print(f'   Mestres após regex      : {len(mestres_pos_regex)}')
    print(f'   ⚙️  Executando fuzzy match...')

    # ── ETAPA 2: Fuzzy match ────────────────────────────────
    freq_mestre = df_prod['pro_txt_mestre'].value_counts().to_dict()
    mapa_fusao = {}   # { variante: canônico }

    for mestre in mestres_pos_regex:
        if mestre in mapa_fusao:
            continue

        resultados = process.extract(
            mestre,
            mestres_pos_regex,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold,
            limit=30
        )

        candidatos = [r[0] for r in resultados]
        canonical = max(candidatos, key=lambda x: freq_mestre.get(x, 0))

        for c in candidatos:
            mapa_fusao[c] = canonical

    df_prod['pro_txt_mestre'] = df_prod['pro_txt_mestre'].map(mapa_fusao)

    df_prod['pro_txt_mestre'] = df_prod['pro_txt_mestre'].fillna(
        df_prod['pro_txt_descricao'].apply(extrair_mestre_regex)
    )

    mestres_pos_fuzzy = df_prod['pro_txt_mestre'].nunique()
    fusoes_realizadas = len(mestres_pos_regex) - mestres_pos_fuzzy
    reducao_total     = ((len(df_prod) - mestres_pos_fuzzy) / len(df_prod)) * 100

    print(f'   Mestres após fuzzy      : {mestres_pos_fuzzy}')
    print(f'   Fusões realizadas       : {fusoes_realizadas}')
    print(f'   Redução total de dimensão: {reducao_total:.2f}%')

    # ── ETAPA 3: Agrupamento final ───────────────────────────
    df_mestre = df_prod.groupby('pro_txt_mestre').agg(
        lista_ids_originais=('pro_id', list),
        exemplo_descricao=('pro_txt_descricao', 'first')
    ).reset_index()

    df_mestre['pro_id_mestre'] = range(5000, 5000 + len(df_mestre))
    print(f'   Entidades mestres geradas: {len(df_mestre)}')

    datasets['TAB_PRODUTO']          = df_prod
    datasets['DATASET_PRODUTO_MESTRE'] = df_mestre

    return datasets

# =========================================================
# EXECUÇÃO DA PLN
# =========================================================

# Ajuste o threshold aqui:
FUZZY_THRESHOLD = 83

datasets = criar_produto_mestre(datasets, threshold=FUZZY_THRESHOLD)

🏗️  PRODUTO MESTRE — REGEX + FUZZY (threshold=83)
   SKUs originais          : 18936
   Mestres após regex      : 2816
   ⚙️  Executando fuzzy match...
   Mestres após fuzzy      : 2306
   Fusões realizadas       : 510
   Redução total de dimensão: 87.82%
   Entidades mestres geradas: 2306


In [0]:
# ============================================================
# CÉLULA 11 — TABELA DE VÍNCULO (DE-PARA SKU → MESTRE)
# ============================================================

def criar_tabela_vinculo(datasets):
    df_mestre = datasets['DATASET_PRODUTO_MESTRE'].copy()

    df_vinculo = df_mestre.explode('lista_ids_originais').rename(
        columns={'lista_ids_originais': 'pro_id'}
    )[['pro_id', 'pro_id_mestre', 'pro_txt_mestre']]

    print(f'🔗 Tabela de vínculo: {len(df_vinculo)} SKUs mapeados para {df_vinculo["pro_id_mestre"].nunique()} mestres.')
    datasets['TAB_VINCULO_MESTRE'] = df_vinculo
    return datasets


datasets = criar_tabela_vinculo(datasets)

🔗 Tabela de vínculo: 18936 SKUs mapeados para 2306 mestres.


In [0]:
# ============================================================
# CÉLULA 12 — PREPARAÇÃO DO INPUT PARA O MODELO
# ============================================================

def preparar_input_final_ia(datasets):
    df_itens   = datasets['TAB_VENDA_ITEM'].copy()
    df_vinculo = datasets['TAB_VINCULO_MESTRE']

    df_final = pd.merge(df_itens, df_vinculo, on='pro_id', how='inner')
    
    # Filtrar apenas vendas válidas (sem vendas zeradas)
    vendas_validas = datasets['TAB_VENDA']['ven_id'].unique()
    df_final = df_final[df_final['ven_id'].isin(vendas_validas)]

    # Presença única: mesma venda + mesmo mestre → conta como 1
    df_input_ia = df_final[['ven_id', 'pro_id_mestre']].drop_duplicates()

    print('=' * 80)
    print('🚀 DATASET PRONTO PARA TREINAMENTO')
    print('=' * 80)
    print(f'   Transações processadas       : {len(df_final)}')
    print(f'   Registros de presença únicos : {len(df_input_ia)}')
    print(f'   Vendas distintas             : {df_input_ia["ven_id"].nunique()}')
    print(f'   Produtos mestres distintos   : {df_input_ia["pro_id_mestre"].nunique()}')

    datasets['TAB_VENDA_ITEM_MESTRE'] = df_final
    datasets['INPUT_IA_RECOMENDACAO'] = df_input_ia
    return datasets


datasets = preparar_input_final_ia(datasets)

🚀 DATASET PRONTO PARA TREINAMENTO
   Transações processadas       : 64858
   Registros de presença únicos : 59528
   Vendas distintas             : 31249
   Produtos mestres distintos   : 1939


In [0]:
# ============================================================
# CÉLULA 13 — TREINAMENTO: FP-GROWTH + REGRAS DE ASSOCIAÇÃO COM MLFLOW
# ============================================================
#
# Por que FP-Growth e não Apriori?
#   O Apriori gera candidatos exaustivamente (O(2^n)).
#   O FP-Growth usa uma árvore compacta (FP-tree) para
#   encontrar os mesmos padrões em O(n), sem gerar candidatos.
#   Mesma interface do mlxtend — troca de uma linha.
#
# Correções sobre a v1 original:
#   1. map() no lugar do depreciado applymap()
#   2. antecedents/consequents preservados como conjuntos completos
#   3. mapear_nome via dicionário O(1) em vez de busca linear em loop
# ============================================================

import asyncio
import uvicorn
import nest_asyncio
import mlflow
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

nest_asyncio.apply()

# ── Configuração do experimento MLflow ──────────────────────
mlflow.set_experiment("/Users/michelcirilo@hotmail.com/FP_Growth_Recomendacao")

# Contador dinâmico (não falha se o servidor do MLflow demorar a atualizar)
try:
    contador_fpgrowth += 1
except NameError:
    contador_fpgrowth = 1


def treinar_ia_mestre(datasets):
    print('=' * 80)
    print('🧠 TREINANDO IA COM FP-GROWTH E MLFLOW')
    print('=' * 80)

    df_input  = datasets['INPUT_IA_RECOMENDACAO']
    df_mestre = datasets['DATASET_PRODUTO_MESTRE']

    # ── Filtrar vendas com ao menos 2 itens ──────────────────
    contagem_itens = df_input.groupby('ven_id').size()
    vendas_validas = contagem_itens[contagem_itens >= 2].index
    df_treino      = df_input[df_input['ven_id'].isin(vendas_validas)].copy()

    qtd_antes  = df_input['ven_id'].nunique()
    qtd_depois = len(vendas_validas)
    print(f'   Vendas de item único removidas : {qtd_antes - qtd_depois}')
    print(f'   Treinando com                  : {qtd_depois} vendas multi-item')

    if df_treino.empty:
        print('❌ Não restaram vendas com mais de um item.')
        return pd.DataFrame()

    # ── Matriz de presença (One-Hot booleana) ────────────────
    basket = (
        df_treino
        .groupby(['ven_id', 'pro_id_mestre'])
        .size()
        .unstack(fill_value=0)
    )

    # map() — substitui o depreciado applymap()
    basket_bool = basket.map(lambda x: True if x >= 1 else False)
    #
    ####################################################################
    ####################################
    ############

    # ── Configuração de Hiperparâmetros ──────────────────────
    min_support_val = 0.0002  # 
    min_threshold_val = 1.3
    metric_val = 'lift'

    ####################################################################
    ###############################
    ##########

    #----------------------------------------------------------------------------------------------------------------------------
    # ── Inicialização do MLflow ──────────────────────────────
    with mlflow.start_run(run_name=f"FP_Growth_Supp_{min_support_val}_Lift_{min_threshold_val}_R{contador_fpgrowth}"):
        
        # 1. Registrar Parâmetros no MLflow
        mlflow.log_param("min_support", min_support_val)
        mlflow.log_param("min_threshold", min_threshold_val)
        mlflow.log_param("metric", metric_val)
        mlflow.log_param("qtd_vendas_treino", qtd_depois)

        # ── FP-Growth ────────────────────────────────────────────
        print('\n   ⚙️  Executando FP-Growth...')
        frequent_itemsets = fpgrowth(
            basket_bool,
            min_support=min_support_val,
            use_colnames=True
        )

        if frequent_itemsets.empty:
            print('⚠️  Nenhum padrão encontrado. Tente reduzir min_support.')
            return pd.DataFrame()

        print(f'   Itemsets frequentes : {len(frequent_itemsets)}')
        
        # 2. Registrar Métricas Intermediárias no MLflow
        mlflow.log_metric("qtd_itemsets_frequentes", len(frequent_itemsets))

        # ── Geração das regras ───────────────────────────────────
        rules = association_rules(
            frequent_itemsets,
            metric=metric_val,
            min_threshold=min_threshold_val  
        )

        # 3. Registrar Métricas Finais do Modelo no MLflow
        mlflow.log_metric("total_regras_geradas", len(rules))
        if not rules.empty:
            mlflow.log_metric("max_lift", float(rules['lift'].max()))
            mlflow.log_metric("mean_confidence", float(rules['confidence'].mean()))

        # ── Dicionário O(1) para lookup de nomes ─────────────────
        mapa_nomes = df_mestre.set_index('pro_id_mestre')['pro_txt_mestre'].to_dict()

        def ids_para_nomes(itemset):
            """Converte frozenset de IDs em string legível."""
            return ' + '.join(mapa_nomes.get(i, str(i)) for i in itemset)

        # ── Monta DataFrame de regras ────────────────────────────
        regras_finais = [
            {
                'busca':         ids_para_nomes(row['antecedents']),
                'recomendado':   ids_para_nomes(row['consequents']),
                'confianca_num': row['confidence'],
                'confianca_txt': f"{round(row['confidence'] * 100, 1)}%",
                'lift':          round(row['lift'], 2),
                'suporte':       round(row['support'], 4),
            }
            for _, row in rules.iterrows()
        ]

        df_recs = pd.DataFrame(regras_finais)
        
        # 4. Registrar o DataFrame final como um Dataset no MLflow
        try:
            dataset_mlflow = mlflow.data.from_pandas(df_recs, name="Regras_Geradas")
            mlflow.log_input(dataset_mlflow, context="training")
        except Exception as e:
            print(f"⚠️ Aviso: Não foi possível registrar o dataset no MLflow ({e})")

        print(f'   ✅ {len(df_recs)} regras de associação geradas e registradas no MLflow.')
        return df_recs


# ── Executar treinamento e construir catálogos ──────────────
db_conhecimento = treinar_ia_mestre(datasets)

if not db_conhecimento.empty:
    df_produtos = datasets['TAB_PRODUTO'].dropna(subset=['pro_txt_descricao', 'pro_txt_mestre'])
    catalogo_detalhado = df_produtos.groupby('pro_txt_mestre')['pro_txt_descricao'].unique().apply(list).to_dict()
    catalogo_busca = df_produtos[['pro_txt_descricao', 'pro_txt_mestre']].rename(columns={'pro_txt_descricao': 'nome', 'pro_txt_mestre': 'mestre'}).drop_duplicates().reset_index(drop=True)
    print(f'\n✅ Produtos indexados  : {len(catalogo_busca)}')
    print(f'✅ Mestres mapeados    : {len(catalogo_detalhado)}')

2026/09/12 17:42:39 INFO mlflow.tracking.fluent: Experiment with name '/Users/michelcirilo@hotmail.com/FP_Growth_Recomendacao' does not exist. Creating a new experiment.


🧠 TREINANDO IA COM FP-GROWTH E MLFLOW
   Vendas de item único removidas : 17662
   Treinando com                  : 13587 vendas multi-item

   ⚙️  Executando FP-Growth...


/databricks/python_shell/lib/dbruntime/safe_oinspect.py:144: DeprecationWarning: `getargspec` function is deprecated as of IPython 7.10and will be removed in future versions.
  argspec = oinspect.getargspec(obj)
/databricks/python_shell/lib/dbruntime/safe_oinspect.py:144: DeprecationWarning: `getargspec` function is deprecated as of IPython 7.10and will be removed in future versions.
  argspec = oinspect.getargspec(obj)


   Itemsets frequentes : 12031
   ✅ 185104 regras de associação geradas e registradas no MLflow.

✅ Produtos indexados  : 6289
✅ Mestres mapeados    : 2305


In [0]:
# ==============================================================================
# SALVAR DATASETS NO CATÁLOGO PARA USO NA REDE NEURAL
# ==============================================================================
# Esta célula persiste todos os dados necessários para o treinamento da rede neural
# no catálogo Unity Catalog, permitindo que as células de ML consumam dados do catálogo
# ao invés de depender de variáveis na memória.
# ==============================================================================

print('=' * 80)
print('💾 SALVANDO DATASETS NO CATÁLOGO UNITY CATALOG')
print('=' * 80)

catalog_name = 'workspace'
schema_name = 'bertoferramentas'

try:
    # 1. INPUT_IA_RECOMENDACAO - Dataset principal para treinamento
    if 'INPUT_IA_RECOMENDACAO' in datasets:
        print('\n📦 Salvando INPUT_IA_RECOMENDACAO...')
        spark_df = spark.createDataFrame(datasets['INPUT_IA_RECOMENDACAO'])
        tabela_destino = f'{catalog_name}.{schema_name}.input_ia_recomendacao'
        spark_df.write.mode('overwrite').saveAsTable(tabela_destino)
        print(f'   ✅ {tabela_destino} - {len(datasets["INPUT_IA_RECOMENDACAO"])} registros')
    else:
        print('   ❌ INPUT_IA_RECOMENDACAO não encontrado')
    
    # 2. TAB_PRODUTO com produto mestre
    if 'TAB_PRODUTO' in datasets:
        print('\n📦 Salvando TAB_PRODUTO...')
        spark_df = spark.createDataFrame(datasets['TAB_PRODUTO'])
        tabela_destino = f'{catalog_name}.{schema_name}.input_ia_tab_produto_processado'
        spark_df.write.mode('overwrite').saveAsTable(tabela_destino)
        print(f'   ✅ {tabela_destino} - {len(datasets["TAB_PRODUTO"])} produtos')
    else:
        print('   ❌ TAB_PRODUTO não encontrado')
    
    # 3. TAB_VINCULO_MESTRE - De-para SKU → Mestre
    if 'TAB_VINCULO_MESTRE' in datasets:
        print('\n📦 Salvando TAB_VINCULO_MESTRE...')
        spark_df = spark.createDataFrame(datasets['TAB_VINCULO_MESTRE'])
        tabela_destino = f'{catalog_name}.{schema_name}.input_ia_tab_vinculo_mestre'
        spark_df.write.mode('overwrite').saveAsTable(tabela_destino)
        print(f'   ✅ {tabela_destino} - {len(datasets["TAB_VINCULO_MESTRE"])} vínculos')
    else:
        print('   ❌ TAB_VINCULO_MESTRE não encontrado')
    
    # 4. DATASET_PRODUTO_MESTRE - Produtos mestres consolidados
    if 'DATASET_PRODUTO_MESTRE' in datasets:
        print('\n📦 Salvando DATASET_PRODUTO_MESTRE...')
        # Explode a lista de IDs originais para salvar no formato tabular
        df_mestre = datasets['DATASET_PRODUTO_MESTRE'].copy()
        df_mestre_exploded = df_mestre.explode('lista_ids_originais')
        spark_df = spark.createDataFrame(df_mestre_exploded)
        tabela_destino = f'{catalog_name}.{schema_name}.input_ia_dataset_produto_mestre'
        spark_df.write.mode('overwrite').saveAsTable(tabela_destino)
        print(f'   ✅ {tabela_destino} - {df_mestre["pro_id_mestre"].nunique()} mestres')
    else:
        print('   ❌ DATASET_PRODUTO_MESTRE não encontrado')
    
    print('\n' + '=' * 80)
    print('✅ TODOS OS DATASETS SALVOS COM SUCESSO NO CATÁLOGO!')
    print('=' * 80)
    print('\n📌 Próximo passo: Execute as células de Rede Neural (19 e 20)')
    print('   As células agora consumirão dados diretamente do catálogo.')
    
except Exception as e:
    print(f'\n❌ Erro ao salvar datasets no catálogo: {e}')
    import traceback
    traceback.print_exc()

/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWar

💾 SALVANDO DATASETS NO CATÁLOGO UNITY CATALOG

📦 Salvando INPUT_IA_RECOMENDACAO...
   ✅ workspace.bertoferramentas.input_ia_recomendacao - 59528 registros

📦 Salvando TAB_PRODUTO...
   ✅ workspace.bertoferramentas.input_ia_tab_produto_processado - 18936 produtos

📦 Salvando TAB_VINCULO_MESTRE...
   ✅ workspace.bertoferramentas.input_ia_tab_vinculo_mestre - 18936 vínculos

📦 Salvando DATASET_PRODUTO_MESTRE...
   ✅ workspace.bertoferramentas.input_ia_dataset_produto_mestre - 2306 mestres

✅ TODOS OS DATASETS SALVOS COM SUCESSO NO CATÁLOGO!

📌 Próximo passo: Execute as células de Rede Neural (19 e 20)
   As células agora consumirão dados diretamente do catálogo.


In [0]:
import os
import pickle
import pandas as pd
import json
from datetime import datetime

# 1. Recupera dinamicamente o e-mail do usuário logado e define o caminho da pasta data do app
try:
    username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
    caminho_salvamento = f"/Workspace/Users/{username}/databricks_apps/Berto_PI6/DatabricsBerto/streamlit-app/data/"
except Exception:
    # Fallback para a pasta compartilhada caso seja executado por um Job automatizado
    caminho_salvamento = "/Workspace/Shared/databricks_apps/data/"

# 2. Cria a pasta no seu Workspace (suporta chamadas normais do Python/OS)
os.makedirs(caminho_salvamento, exist_ok=True)

print(f"⏳ Salvando artefatos diretamente no seu Workspace: {caminho_salvamento}")

# 3. Salva os DataFrames de conhecimento e busca em formato Parquet
# Garante que são DataFrames pandas puros antes de salvar
pd.DataFrame(db_conhecimento).to_parquet(
    os.path.join(caminho_salvamento, "db_conhecimento.parquet"), 
    index=False
)
pd.DataFrame(catalogo_busca).to_parquet(
    os.path.join(caminho_salvamento, "catalogo_busca.parquet"), 
    index=False
)

# 4. Salva o dicionário detalhado do catálogo usando Pickle
with open(os.path.join(caminho_salvamento, "catalogo_detalhado.pkl"), "wb") as f:
    pickle.dump(catalogo_detalhado, f)

# 5. Salva metadados com data de atualização
metadata = {
    "data_atualizacao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_regras": len(db_conhecimento),
    "total_produtos": len(catalogo_busca),
    "total_mestres": len(catalogo_detalhado)
}
with open(os.path.join(caminho_salvamento, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("\n✅ Sucesso! Os arquivos foram salvos na pasta data do app Streamlit.")
print(f"📂 Localização: {caminho_salvamento}")
print(f"📅 Data de atualização: {metadata['data_atualizacao']}")

⏳ Salvando artefatos diretamente no seu Workspace: /Workspace/Users/michelcirilo@hotmail.com/databricks_apps/Berto_PI6/DatabricsBerto/streamlit-app/data/


/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWar


✅ Sucesso! Os arquivos foram salvos na pasta data do app Streamlit.
📂 Localização: /Workspace/Users/michelcirilo@hotmail.com/databricks_apps/Berto_PI6/DatabricsBerto/streamlit-app/data/
📅 Data de atualização: 2026-09-12 17:48:38
